In [ ]:
using ProgressBars
include("/Users/travail/Desktop/Quantic/Dev/Stage_quantic_2025/Code_Julia/Julia_file/classical_oscillators.jl");

# Emission spectrum for the two-photon process

In [ ]:
using Bessels

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.7*2*pi
ω_b_0 = 7.05*2*pi

Lamb_shift = - 0.05 #Lamb shift in GHz

ω_b = ω_b_0 #+ Lamb_shift*2*pi # Shifted frequency of mode b

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.2
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37*2*pi




κ=1/10; # 1/ns from Jeffrey et al.
ϕ_quantum=1/2

ω_d=ω_b_0 # Drive frequency
ω_p=2ω_a-ω_d # Pump frequency

ϵ_0 = 0.25
g2_max = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10*g2_max

κ_2 = 4*g2_max^2/κ;

In [ ]:
n=7000
t_max=15/κ

bath=Exp_decay
bath_params = bath_params=(ω_0=50*2*pi, ω_cut_off=30*2*pi)

AB, J = bath(bath_params=bath_params,κ=κ, C_inv=C_b_inv)
chain_params = chaincoeffs_finiteT(n, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=7000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

In [ ]:
gq=vcat(- chain_params[2])
gϕ=vcat(chain_params[2])

Mq_chain=SymTridiagonal(chain_params[1], gq)
Mϕ_chain=SymTridiagonal(chain_params[1], gϕ)

In [ ]:
ϕ,ϕ_dot,t_list,_,_ = two_photon_1d_chain(n, t_max; chain_params=chain_params,
     circuit_params=(ω_a=ω_a,ω_b=ω_b,E_J =E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0,κ=κ, ϵ_d=ϵ_d, ω_d=ω_b_0),
     init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞,minimal_a=false,emission=true)

In [ ]:
ϕ_a,ϕ_b,q_a,q_b,t_list = two_photon_1d_chain(n, t_max;chain_params=chain_params,
     circuit_params=(ω_a=ω_a,ω_b=ω_b,E_J =E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0,κ=κ, ϵ_d=ϵ_d, ω_d=ω_b_0+Lamb_shift*2*pi),
     init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞,minimal_a=false,emission=false)

In [ ]:
plot(t_list*κ,ϕ_b)

In [ ]:
Re_a = ϕ_a / sqrt(2*Z_a)
Im_a = q_a * sqrt(Z_a/2)

ϕ_rotating_a =@. ϕ_a * cos(ω_a * t_list) + q_a*Z_a* sin(-ω_a * t_list)
q_rotating_a =@. -ϕ_a/Z_a * sin(-ω_a * t_list) + q_a* cos(ω_a * t_list)

Re_rotating_a = ϕ_rotating_a / sqrt(2*Z_a)
Im_rotating_a = q_rotating_a * sqrt(Z_a/2)

# ϕ_rotating_b =@. ϕ_CL_b * cos(ω_a * t_list) + q_CL_b*Z_a* sin(-ω_a * t_list)
# q_rotating_b =@. -ϕ_CL_b/Z_a * sin(-ω_a * t_list) + q_CL_b* cos(ω_a * t_list);
p1=plot(Re_a,Im_a)
p2=plot(Re_rotating_a,Im_rotating_a)

layout = (1,2)
plot(p1,p2,layout=layout, xlabel=L"\Re(a)", ylabel=L"\Im(a)", title=["Resonator a" "Resonator a RF"], legend=false,
size=(layout[2], layout[1]).*size_px_car,
margin = 10Plots.mm)
#savefig("cat_state_chain.pdf")

In [ ]:
A_chain = Mq_chain * Mϕ_chain

vals_chain, vecs_chain = eigen(A_chain)
ωk²_chain = real(vals_chain)
ωk_chain = sqrt.(ωk²_chain)
P_chain = real(vecs_chain)


Dq_chain = diag(P_chain'*(Mq_chain \ P_chain))  # uses linear solve instead of inv
Dϕ_chain = diag(P_chain' * Mϕ_chain * P_chain)

In [ ]:
# Solve P_chain * S = ϕ for S instead of inverting
S = P_chain \ ϕ
S_dot = P_chain \ ϕ_dot

# Energy in normal modes
Energies_normal_modes_chain = Dq_chain .* S_dot.^2 .+ Dϕ_chain .* S.^2;

In [ ]:
# Solve P_chain * S = ϕ for S instead of inverting
S_8 = P_chain \ ϕ_8
S_dot_8 = P_chain \ ϕ_dot_8

# Energy in normal modes
Energies_normal_modes_chain_8 = Dq_chain .* S_dot_8.^2 .+ Dϕ_chain .* S_8.^2;

In [ ]:
# Solve P_chain * S = ϕ for S instead of inverting
S_6 = P_chain \ ϕ_6
S_dot_6 = P_chain \ ϕ_dot_6

# Energy in normal modes
Energies_normal_modes_chain_6 = Dq_chain .* S_dot_6.^2 .+ Dϕ_chain .* S_6.^2;

In [ ]:
(2*ω_a- ω_b)/2pi

In [ ]:
3f_p

In [ ]:
f_a=ω_a/2pi
f_b=ω_b/2pi
f_p=abs(ω_p/2pi)
plot(ωk_chain/2pi,Energies_normal_modes_chain)
vline!([f_a],linestyle=:dash, color=:red)
annotate!((f_a)-.2,1e-0, Plots.text(L"\omega_d/2+\omega_p/2",18, :red, rotation = 90 ))
vline!([f_a-2f_p],linestyle=:dash, color=:red)
annotate!((f_a-2f_p)-.2,1e-0, Plots.text(L"\omega_d/2-3\omega_p/2",18, :red, rotation = 90 ))
vline!([f_a+2f_p],linestyle=:dash, color=:red)
annotate!((f_a+2f_p)-.2,1e-0, Plots.text(L"\omega_d/2+5\omega_p/2",18, :red, rotation = 90 ))
vline!([f_a-4f_p],linestyle=:dash, color=:red)
annotate!((f_a-4f_p)-.2,1e-0, Plots.text(L"\omega_d/2-7\omega_p/2",18, :red, rotation = 90 ))
vline!([f_a+4f_p],linestyle=:dash, color=:red)
annotate!((f_a+4f_p)-.2,1e-0, Plots.text(L"\omega_d/2+9\omega_p/2",18, :red, rotation = 90 ))

vline!([f_p],linestyle=:dash, color=:red)
annotate!(f_p-.2,1e-0, Plots.text(L"\omega_p", 18, :red, rotation = 90 ))
vline!([3f_p],linestyle=:dash, color=:red)
annotate!(3f_p-.2,1e-0, Plots.text(L"3\omega_p", 18, :red, rotation = 90 ))
vline!([f_b],linestyle=:dash, color=:red)

annotate!(f_b-.2,1e-8, Plots.text(L"\omega_d",18, :red, rotation = 90 ))
vline!([f_b-2f_p],linestyle=:dash, color=:red)
annotate!((f_b-2f_p)-.2,1e-8, Plots.text(L"\omega_d-2\omega_p",18, :red, rotation = 90 ))
vline!([f_b+2f_p],linestyle=:dash, color=:red)
annotate!((f_b+2f_p)-.2,1e-8, Plots.text(L"\omega_d+2\omega_p",18, :red, rotation = 90 ))
#vline!([f_b-4f_p],linestyle=:dash, color=:red)
#annotate!((f_b-4f_p)-.2,1e-8, Plots.text(L"\omega_d-4\omega_p",18, :red, rotation = 90 ))
vline!([f_b+4f_p],linestyle=:dash, color=:red)
annotate!((f_b+4f_p)-.2,1e-8, Plots.text(L"\omega_d+4\omega_p",18, :red, rotation = 90 ))

vline!([3f_b/2-5f_p/2],linestyle=:dash, color=:red)
annotate!((3f_b/2-5f_p/2)-.2,1e-8, Plots.text(L"3\omega_d/2-5\omega_p/2",18, :red, rotation = 90 ))
vline!([3f_b/2-f_p/2],linestyle=:dash, color=:red) 
annotate!((3f_b/2-f_p/2)-.2,1e-8, Plots.text(L"3\omega_d/2-\omega_p/2",18, :red, rotation = 90 ))
vline!([3f_b/2+3f_p/2],linestyle=:dash, color=:red)
annotate!((3f_b/2+3f_p/2)-.2,1e-8, Plots.text(L"3\omega_d/2+3\omega_p/2",18, :red, rotation = 90 ))
vline!([3f_b/2+7f_p/2],linestyle=:dash, color=:red)
annotate!((3f_b/2+7f_p/2)-.2,1e-8, Plots.text(L"3\omega_d/2+7\omega_p/2",18, :red, rotation = 90 ))

#vline!([2f_b-3f_p],linestyle=:dash, color=:red)
#annotate!((2f_b-3f_p)-.2,1e-8, Plots.text(L"2\omega_d-3\omega_p",18, :red, rotation = 90 ))
vline!([2f_b-f_p],linestyle=:dash, color=:red)
annotate!((2f_b-f_p)-.2,1e-8, Plots.text(L"2\omega_d-\omega_p",18, :red, rotation = 90 ))
vline!([2f_b+f_p],linestyle=:dash, color=:red)
annotate!((2f_b+f_p)-.2,1e-8, Plots.text(L"2\omega_d+\omega_p",18, :red, rotation = 90 ))

plot!(ylabel="Energy stored in the chain", xlabel="Frequency (GHz)", title = L"κt = "*"$(κ*t_max), "*L"n_{osc}="*"$(n)", legend=false, yaxis=:log, xlims=(0.1, 16), ylims=(1e-10, 1e2))
#savefig("../../Figures/Emission_spectrum/Energies_normal_modes_chain_two_photon_09_07.pdf")

In [ ]:
f_b/2+9*f_p/2

In [ ]:
plot(ωk_chain/2pi,Energies_normal_modes_chain)
plot!(ωk_chain/2pi,Energies_normal_modes_chain_8)
plot!(ωk_chain/2pi,Energies_normal_modes_chain_6)
plot!(ylabel="Energy stored in the chain", xlabel="Frequency (GHz)", title = L"κt = "*"$(κ*t_max), "*L"n_{osc}="*"$(n)", legend=false, yaxis=:log, xlims=(0.1, 16), ylims=(1e-10, 1e2))

In [ ]:
ϵ_d_list=(4:0.5:15) .*g2_max
Energies_normal_modes_chain = Array{Float64}(undef, length(ϵ_d_list), length(ωk_chain))

for i in ProgressBar(1:length(ϵ_d_list))
    ϵ_d_i=ϵ_d_list[i]
    ϕ_i,ϕ_dot_i,_,_,_ = two_photon_1d_chain(n, t_max; chain_params=chain_params,
     circuit_params=(ω_a=ω_a,ω_b=ω_b,E_J =E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0,κ=κ, ϵ_d=ϵ_d_i, ω_d=ω_b_0),
     init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞,minimal_a=false,emission=true)
    
    # Solve P_chain * S = ϕ for S instead of inverting
    S_i = P_chain \ ϕ_i
    S_dot_i = P_chain \ ϕ_dot_i

    # Energy in normal modes
    Energies_normal_modes_chain[i,:] = Dq_chain .* S_dot_i.^2 .+ Dϕ_chain .* S_i.^2;
end

In [ ]:
heatmap(ωk_chain/2pi,ϵ_d_list./g2_max./2,log10.(Energies_normal_modes_chain))
plot!(ylabel="Cat size", xlabel="Frequency (GHz)",colorbar_title="\n"*L"\log_{10}"*"(Energy)",
title = L"κt = "*"$(κ*t_max), "*L"n_{osc}="*"$(n)",
legend=false,
xlims=(0.1, 15.4),clims=(-10, 2),
right_margin=30mm,)
savefig("../../Figures/Emission_spectrum/Energies_normal_modes_chain_two_photon_heatmap_09_07.pdf")

In [ ]:
ϕ_off,ϕ_dot_off,t_list,Mq_chain,Mϕ_chain = two_photon_1d_chain(n, t_max; bath = bath, bath_params = bath_params,
     circuit_params=(ω_a=ω_a,ω_b=ω_b,E_J =E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0,κ=κ, ϵ_d=ϵ_d, ω_d=ω_b_0+Lamb_shift*2*pi),
     init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞,minimal_a=false,emission=true)

In [ ]:
# Solve P_chain * S = ϕ for S instead of inverting
S = P_chain \ ϕ_off
S_dot = P_chain \ ϕ_dot_off

# Energy in normal modes
Energies_normal_modes_chain_off = Dq_chain .* S_dot.^2 .+ Dϕ_chain .* S.^2;

In [ ]:
plot(ωk_chain/2pi,Energies_normal_modes_chain_off)
vline!([ω_a/2pi],linestyle=:dash, color=:red)
vline!([ω_b/2pi],linestyle=:dash, color=:red)
vline!([ω_p/2pi],linestyle=:dash, color=:red)
annotate!(ω_a/2pi-.2,1e-0, Plots.text("mode "*L"a", 20, :red, rotation = 90 ))
annotate!(ω_b/2pi-.2,1e-8, Plots.text("mode "*L"b"*" (shifted)",20, :red, rotation = 90 ))
annotate!(ω_p/2pi-.2,1e-0, Plots.text("pump", 20, :red, rotation = 90 ))
plot!(ylabel="Energy stored in the chain", xlabel="Frequency (GHz)", title = L"κt = "*"$(κ*t_max), "*L"n_{osc}="*"$(n)", legend=false, yaxis=:log, xlims=(0.1, 9.9), ylims=(1e-10, 1e2))
#savefig("../Figures/Emission_spectrum/Energies_normal_modes_chain_two_photon_07_07.pdf")

In [ ]:
Energies_normal_modes_chain = Matrix{Float64}(undef,(3,size(ϕ_chain,2)))

# Compute S_chain and S_chain_dot on the fly, avoid storing large intermediates
for t in size(ϕ_chain, 1)-2:size(ϕ_chain, 1)
    S = P_chain_inv * ϕ_chain[t,:]
    S_dot = P_chain_inv * ϕ_dot_chain[t,:]
    Energies_normal_modes_chain[t-(size(ϕ_chain, 1)-2)+1,:] = Dq_chain .* S_dot.^2 .+ Dϕ_chain .* S.^2
end